In [ ]:
import sys
from pathlib import Path

import numpy as np
import netCDF4 as nc
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths

## Configuration

In [ ]:
# Which split to load (0–8)
SPLIT_IDX = 0

# Which partition to plot: 'tr', 'va', or 'te'
PARTITION = 'tr'

# Which sample index within that partition to plot
SAMPLE_IDX = 0

## Load grid and split file

In [ ]:
# ------------------------------------------------------------------
# Grid  (lat / lon on the CESM2 POP ocean grid)
# ------------------------------------------------------------------
grid_path = paths.CESM2LE_SST_DIR / 'grid' / 'cesm2le_sst_grid.nc'

with nc.Dataset(str(grid_path), 'r') as ds_grid:
    lat = np.array(ds_grid.variables['lat'][:],  dtype=np.float32)   # (nx, ny)
    lon = np.array(ds_grid.variables['lon'][:], dtype=np.float32)   # (nx, ny)

print(f'Grid shape : {lat.shape}')

In [ ]:
# ------------------------------------------------------------------
# TVT split file
# ------------------------------------------------------------------
split_path = paths.tvt_split_path(SPLIT_IDX)
print(f'Loading : {split_path}')

ds = xr.open_dataset(split_path)
print(ds)

mu    = float(ds['mu_train'])
sigma = float(ds['sigma_train'])
print(f'\nmu_train    = {mu:.4f} °C')
print(f'sigma_train = {sigma:.4f} °C')

In [ ]:
# ------------------------------------------------------------------
# Partition summary
# ------------------------------------------------------------------
for part, label_key, sst_key in [
        ('train', 'slow_tr', 'sst_tr'),
        ('val',   'slow_va', 'sst_va'),
        ('test',  'slow_te', 'sst_te')]:
    labels = ds[label_key].values
    n_slow = labels.sum()
    print(f'  {part:5s}  n={len(labels):5d}   slowdown={n_slow:4d} ({100*n_slow/len(labels):.1f}%)')

## Map of one SST sample

In [ ]:
# ------------------------------------------------------------------
# Plot standardised SST (land sentinel = -10 masked out)
# ------------------------------------------------------------------
sst_key = f'sst_{PARTITION}'
sst_std = ds[sst_key].values[SAMPLE_IDX]          # (nx, ny)

label_key = f'slow_{PARTITION}'
label = int(ds[label_key].values[SAMPLE_IDX])
label_str = 'slowdown' if label == 1 else 'no slowdown'

# Mask land sentinel
sst_plot = np.ma.masked_where(sst_std <= -9, sst_std)

fig = plt.figure(figsize=(11, 5))
ax  = plt.axes(projection=ccrs.PlateCarree())

ax.set_global()
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)
ax.coastlines(linewidth=0.5, color='k', zorder=3)

im = ax.pcolormesh(
    lon, lat, sst_plot,
    vmin=-3, vmax=3,
    cmap=cmocean.cm.balance,
    shading='auto',
    transform=ccrs.PlateCarree(),
)

gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
gl.top_labels   = False
gl.right_labels = False

ax.set_title(
    f'Split {SPLIT_IDX}  |  partition={PARTITION}  |  sample={SAMPLE_IDX}  |  label={label} ({label_str})\n'
    f'Standardised JJA SST  (μ={mu:.2f} °C, σ={sigma:.2f} °C)',
    fontsize=11, pad=6,
)

cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.08, shrink=0.8)
cbar.set_label('Standardised SST anomaly  (σ units)', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Same sample back in physical units  (°C anomaly from ensemble mean)
# ------------------------------------------------------------------
sst_phys = np.ma.masked_where(sst_std <= -9, sst_std * sigma + mu)

fig = plt.figure(figsize=(11, 5))
ax  = plt.axes(projection=ccrs.PlateCarree())

ax.set_global()
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)
ax.coastlines(linewidth=0.5, color='k', zorder=3)

im = ax.pcolormesh(
    lon, lat, sst_phys,
    vmin=-5, vmax=5,
    cmap=cmocean.cm.balance,
    shading='auto',
    transform=ccrs.PlateCarree(),
)

gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
gl.top_labels   = False
gl.right_labels = False

ax.set_title(
    f'Split {SPLIT_IDX}  |  partition={PARTITION}  |  sample={SAMPLE_IDX}  |  label={label} ({label_str})\n'
    f'JJA SST anomaly (physical units, °C)',
    fontsize=11, pad=6,
)

cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.08, shrink=0.8)
cbar.set_label('JJA SST anomaly  (°C)', fontsize=10)

plt.tight_layout()
plt.show()